# 市場客群 KMeans 分群分析
此 Notebook 對 Mall_Customers.csv 進行 KMeans 分群，包含 Elbow、Silhouette 及 PCA 可視化。

In [ ]:
# 1. 從 GitHub Clone 資料（只需執行一次）
import os

if not os.path.exists('DataMiningExercises'):
    !git clone https://github.com/JackPan0521/DataMiningExercises.git
else:
    !git -C DataMiningExercises pull
print('Done')

In [ ]:
# 2. 安裝套件（Colab 通常已內建，視需求執行）
# !pip install scikit-learn pandas matplotlib numpy

In [ ]:
# 3. Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
# 4. 參數設定（修改這裡來調整分析行為）
CHOSEN_K    = None            # 手動指定 K；設為 None 則自動選 K
K_MIN       = 2               # 自動搜尋最小 K
K_MAX       = 12              # 自動搜尋最大 K
AUTO_K_MODE = 'capped-silhouette'  # 'silhouette' | 'elbow' | 'capped-silhouette'
AUTO_K_CAP  = 8               # capped-silhouette 的上限

In [ ]:
# 5. 載入資料
csv_path = 'DataMiningExercises/市場客群/Mall_Customers.csv'

last_error = None
for enc in ['utf-8-sig', 'cp950', 'utf-8']:
    try:
        df = pd.read_csv(csv_path, encoding=enc)
        print(f'Loaded {len(df)} rows with encoding: {enc}')
        break
    except Exception as exc:
        last_error = exc
else:
    raise RuntimeError(f'Failed to read CSV; last error: {last_error}')

print('\n--- Raw dtypes ---')
print(df.dtypes)
df.head(3)

In [ ]:
# 6. 前處理
numeric_cols  = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
category_cols = ['Genre']

model_df = df[numeric_cols + category_cols].copy()
label_encoder = LabelEncoder()
for col in category_cols:
    model_df[col] = label_encoder.fit_transform(model_df[col].astype(str))

model_df = model_df.fillna(model_df.median(numeric_only=True))
scaler = StandardScaler()
X = scaler.fit_transform(model_df)
print('Feature matrix shape:', X.shape)

In [ ]:
# 7. K 值搜尋（Elbow + Silhouette）
if K_MIN < 2 or K_MAX < K_MIN:
    raise ValueError('Invalid k range: ensure K_MIN >= 2 and K_MAX >= K_MIN')

inertias, silhouettes = [], []

for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=min(3000, len(X)), random_state=42))
    print(f'  k={k:2d}  inertia={km.inertia_:,.2f}  silhouette={silhouettes[-1]:.4f}')

k_values     = np.array(list(range(K_MIN, K_MAX + 1)))
sil_array    = np.array(silhouettes)
inertia_array = np.array(inertias)
sil_best_k   = int(k_values[np.argmax(sil_array)])

# Elbow K
if len(inertia_array) >= 3:
    second_diff = np.diff(inertia_array, n=2)
    elbow_index = int(np.argmax(np.abs(second_diff))) + 2
    elbow_k = int(k_values[elbow_index])
else:
    elbow_k = sil_best_k

# Capped silhouette K
cap_k    = max(K_MIN, min(AUTO_K_CAP, K_MAX))
cap_mask = k_values <= cap_k
capped_best_k = int(k_values[cap_mask][np.argmax(sil_array[cap_mask])]) if np.any(cap_mask) else sil_best_k

# Auto-K 策略
if AUTO_K_MODE == 'silhouette':
    auto_best_k = sil_best_k
elif AUTO_K_MODE == 'elbow':
    auto_best_k = elbow_k
else:
    auto_best_k = capped_best_k

if CHOSEN_K is not None:
    chosen_k = CHOSEN_K
    print(f'\nUser-selected K={chosen_k} (auto best K={auto_best_k}, mode={AUTO_K_MODE}, cap={cap_k})')
else:
    chosen_k = auto_best_k
    print(f'\nAuto-selected K={chosen_k} (mode={AUTO_K_MODE}, silhouette_best={sil_best_k}, elbow={elbow_k}, cap_best={capped_best_k})')

In [ ]:
# 8. 最終分群
final_km = KMeans(n_clusters=chosen_k, n_init=20, random_state=42)
df['Cluster'] = final_km.fit_predict(X)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())

print('\n--- Cluster Profile (numeric means) ---')
profile = df.groupby('Cluster')[numeric_cols].mean().round(2)
print(profile.to_string())

print('\n--- Dominant Genre per cluster ---')
dominant_genre = df.groupby('Cluster')['Genre'].agg(lambda x: x.value_counts().idxmax())
print(dominant_genre.to_string())

In [ ]:
# 9. 三圖視覺化（Elbow / Silhouette / PCA 2D）
k_range = list(range(K_MIN, K_MAX + 1))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow
axes[0].plot(k_range, inertias, marker='o', color='steelblue')
axes[0].set_title('Elbow Curve (Inertia)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(auto_best_k, color='red',    linestyle='--', label=f'Auto best k={auto_best_k}')
axes[0].axvline(elbow_k,     color='green',  linestyle=':',  label=f'Elbow k={elbow_k}')
if CHOSEN_K is not None:
    axes[0].axvline(chosen_k, color='purple', linestyle='-.', label=f'Chosen k={chosen_k}')
axes[0].legend()

# Silhouette
axes[1].plot(k_range, silhouettes, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(auto_best_k, color='red',   linestyle='--', label=f'Auto best k={auto_best_k}')
axes[1].axvline(sil_best_k,  color='green', linestyle=':',  label=f'Silhouette best k={sil_best_k}')
if CHOSEN_K is not None:
    axes[1].axvline(chosen_k, color='purple', linestyle='-.', label=f'Chosen k={chosen_k}')
axes[1].legend()

# PCA 2D
pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
scatter = axes[2].scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=df['Cluster'], cmap='tab10', alpha=0.75, s=40
)
axes[2].set_title(f'PCA 2D - k={chosen_k} clusters')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[2].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# 10. 儲存結果並下載（選用）
out_csv = 'kmeans_market_results.csv'
df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'Saved: {out_csv}')

# 儲存圖片
out_png = 'kmeans_market_segment.png'
fig.savefig(out_png, dpi=160)
print(f'Saved: {out_png}')

# 下載到本機（Colab 限定）
try:
    from google.colab import files
    files.download(out_csv)
    files.download(out_png)
except ImportError:
    print('非 Colab 環境，略過下載步驟')